In [18]:
import torch
import torchvision
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torch.autograd import Variable
from torch import optim

import torch.nn as nn
import torch.nn.functional as F
import os
import cv2
import time
from PIL import Image
from tqdm import tqdm_notebook as tqdm
import random
from matplotlib import pyplot as plt

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [19]:
# 이미지 전처리 클래스
# 학습 시에는 데이터 증강을 수행하고, 검증 시에는 이미지 크기만 조정한다.
class ImageTransform():
  def __init__(self, resize, mean, std):
    self.data_transform = {
        # 학습: 모델이 다양한 상황에서 적응하도록 이미지를 무작위로 변형
        'train': transforms.Compose([
            transforms.RandomResizedCrop(resize, scale=(0.5, 1.0)), # RandomResizedCrop: 주어진 크기로 조정(reszie: 224x224)
            transforms.RandomHorizontalFlip(),  # 50% 확률로 이미지를 좌우로 뒤집음
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ]),
        # 검증: 실제 성능을 평가하므로 무작위 변형 없이 일정한 가공만 수행
        'val': transforms.Compose([
            transforms.Resize(256), # 짧은 쪽의 길이를 256으로 변경
            transforms.CenterCrop(resize),  # 이미지의 중앙부를 resize 크기만큼 잘라냄
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ])
    }

  def __call__(self, img, phase):
    return self.data_transform[phase](img)

In [17]:
cat_directory = r'/content/drive/MyDrive/Colab Notebooks/LeNet/Cat'
dog_directory = r'/content/drive/MyDrive/Colab Notebooks/LeNet/Dog'

cat_images_filepaths = sorted([os.path.join(cat_directory, f) for f in os.listdir(cat_directory)])
dog_images_filepaths = sorted([os.path.join(dog_directory, f) for f in os.listdir(dog_directory)])
images_filepaths = [*cat_images_filepaths, *dog_images_filepaths]

# 데이터 무결성 검사: OpenCV로 읽었을 때 None이 아닌(손상되지 않은) 파일 경로만 추출
correct_images_filepaths = [i for i in images_filepaths if cv2.imread(i) is not None]

random.seed(42)
random.shuffle(correct_images_filepaths)

# 데이터셋 분할
train_images_filepaths = correct_images_filepaths[:400]
val_images_filepaths = correct_images_filepaths[400:-10]
test_images_filepaths = correct_images_filepaths[-10:]

print(len(train_images_filepaths), len(val_images_filepaths), len(test_images_filepaths))

400 92 10


In [20]:
def display_image_grid(images_filepaths, predicted_labels=(), cols=5):
  rows = len(images_filepaths) // cols
  figure, ax = plt.subplots(nrows=rows, ncols=cols, figsize=(12, 6))

  for i, image_filepath in enumerate(images_filepaths):
    # 이미지 로드 및 색생 공간 변환
    image = cv2.imread(image_filepath)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # COLOR_BGR2RGB: BGR 채널 이미지를 RGB로 변경

    true_label = os.path.normpath(image_filepath).split(os.sep)[-2]

    predicted_label = predicted_labels[i] if predicted_labels else true_label

    color = "green" if true_label == predicted_label else "red"

    # ax.ravel(): 다차원 배열인 ax를 1차원으로 평탄화하여 인덱스 접근을 쉽게 한다.
    ax.ravel()[i].imshow(image)
    ax.ravel()[i].set_title(predicted_label, color=color)
    ax.ravel()[i].set_axis_off()

  plt.tight_layout()
  plt.show()

In [ ]:
display_image_grid(test_images_filepaths)

In [22]:
class DogvsCatDataset(Dataset):
  # 데이터셋의 전처리(데이터 변형 적용)
  def __init__(self, file_list, transform=None, phase='train'):
    self.file_list = file_list
    self.transform = transform # 인자 이름은 transforms였으나 내부 사용은 transform
    self.phase = phase  # train 적용

  def __len__(self):
    return len(self.file_list)

  # 데이터셋에서 데이터를 가져오는 부분으로 결과는 텐서 형태가 된다.
  def __getitem__(self, idx):
    img_path = self.file_list[idx]
    img = Image.open(img_path)
    img_transformed = self.transform(img, self.phase) # 이미지에 train 전처리를 적용
    label = img_path.split('/')[-1].split('.')[0]

    if label == 'dog':
      label = 1
    elif label == 'cat':
      label = 0

    return img_transformed, label

In [23]:
size = 224
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
batch_size = 32

In [24]:
train_dataset = DogvsCatDataset(train_images_filepaths, transform=ImageTransform(size, mean, std), phase='train') # 훈련 이미지에 train_transform를 적용
val_dataset = DogvsCatDataset(val_images_filepaths, transform=ImageTransform(size, mean, std), phase='val') # 검증 이미지에 test_transform를 적용

index = 0
print(train_dataset.__getitem__(index)[0].size()) # 훈련 데이터(train_dataset.__getitem__[0][0])의 크기(size()) 출력
print(train_dataset.__getitem__(index)[1]) # 훈련 데이터의 레이블 출력

torch.Size([3, 224, 224])
0


In [25]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
dataloader_dict = {'train': train_dataloader, 'val': val_dataloader}

batch_iterator = iter(train_dataloader)
inputs, label = next(batch_iterator)
print(inputs.size())
print(label)

torch.Size([32, 3, 224, 224])
tensor([0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0,
        0, 1, 0, 0, 1, 0, 0, 0])


In [26]:
class LeNet(nn.Module):
  def __init__(self):
    super(LeNet, self).__init__()

    # 입력 형태는 (3, 224, 224)가 되며 출력 형태는 (weight-kernel_size+1)/stride에 따라 (16,220, 220)
    self.cnn1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=0)
    self.relu1 = nn.ReLU()

    # 최대 풀링 적용 이후 출력 형태는 220/2가 되어 (16, 110, 110)
    self.maxpool1 = nn.MaxPool2d(kernel_size=2)

    # 또 다시 2D 합성곱층이 적용되며 출력 형태는 (32, 106, 106)
    self.cnn2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride=1, padding=0)
    self.relu2 = nn.ReLU()

    # 최대 풀링이 적용되며 출력 향태는 (32, 53, 53)
    self.maxpool2 = nn.MaxPool2d(kernel_size=2)

    self.fc1 = nn.Linear(32*53*53, 512)
    self.relu5 = nn.ReLU()
    self.fc2 = nn.Linear(512, 2)
    self.output = nn.Softmax(dim=1)

  def forward(self, x):
    out = self.cnn1(x)
    out = self.relu1(out)
    out = self.maxpool1(out)
    out = self.cnn2(out)
    out = self.relu2(out)
    out = self.maxpool2(out)

    # 완전연결층에 데이터를 전달하기 위해 데이터 형태를 1차원으로 바꾼다.
    out = out.view(out.size(0), -1)
    out = self.fc1(out)
    out = self.fc2(out)
    out = self.output(out)
    return out

In [27]:
model = LeNet()
print(model)

LeNet(
  (cnn1): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1))
  (relu1): ReLU()
  (maxpool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (cnn2): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1))
  (relu2): ReLU()
  (maxpool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=89888, out_features=512, bias=True)
  (relu5): ReLU()
  (fc2): Linear(in_features=512, out_features=2, bias=True)
  (output): Softmax(dim=1)
)


In [28]:
# summary를 통해 모델의 네트워크 관련 정보를 확인할 수 있다.
from torchsummary import summary
summary(model, input_size=(3, 224, 224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 16, 220, 220]           1,216
              ReLU-2         [-1, 16, 220, 220]               0
         MaxPool2d-3         [-1, 16, 110, 110]               0
            Conv2d-4         [-1, 32, 106, 106]          12,832
              ReLU-5         [-1, 32, 106, 106]               0
         MaxPool2d-6           [-1, 32, 53, 53]               0
            Linear-7                  [-1, 512]      46,023,168
            Linear-8                    [-1, 2]           1,026
           Softmax-9                    [-1, 2]               0
Total params: 46,038,242
Trainable params: 46,038,242
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 19.47
Params size (MB): 175.62
Estimated Total Size (MB): 195.67
--------------------------------

In [29]:
def count_parameters(model):
  # p.numel(): 파라미터 텐서 내의 요소(element) 총 개수를 반환(예: 3x3 커널이면 9개)
  return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

The model has 46,038,242 trainable parameters


In [30]:
# momentum: SGD를 적절한 방향으로 가속화하며 흔들림(진동)을 줄여주는 매개변수
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
criterion = nn.CrossEntropyLoss()

In [31]:
model = model.to(device)
criterion = criterion.to(device)

In [32]:
def train_model(model, dataloader_dict, criterion, optimizer, num_epoch):
  since = time.time()
  best_acc = 0.0

  for epoch in range(num_epoch):
    print('Epoch {}/{}'.format(epoch+1, num_epoch))
    print('-'*20)

    for phase in ['train', 'val']:
      if phase == 'train':
        model.train()
      else:
        model.eval()

      epoch_loss = 0.0
      epoch_corrects = 0

      # dataloader_dict: 훈련 데이터셋(train_loader)을 의미
      for inputs, labels in tqdm(dataloader_dict[phase]):
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()

        with torch.set_grad_enabled(phase == 'train'):
          outputs = model(inputs)
          _, preds = torch.max(outputs, 1)
          loss = criterion(outputs, labels)

          if phase == 'train':
            loss.backward()
            optimizer.step()

          epoch_loss += loss.item() * inputs.size(0)
          epoch_corrects += torch.sum(preds == labels.data)

      epoch_loss = epoch_loss / len(dataloader_dict[phase].dataset)
      epoch_acc = epoch_corrects.double() / len(dataloader_dict[phase].dataset)

      print('{} Loss: {:.4f} Acc: {:.4f}'.format(phase, epoch_loss, epoch_acc))

      if phase == 'val' and epoch_acc > best_acc:
        best_acc = epoch_acc
        best_model_wts = model.state_dict()

  time_elapsed = time.time() - since
  print('Training complete in {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))
  print('Best val Acc: {:4f}'.format(best_acc))
  return model

In [ ]:
num_epoch = 10
model = train_model(model, dataloader_dict, criterion, optimizer, num_epoch)

In [35]:
import pandas as pd

id_list = []
pred_list = []
_id = 0
with torch.no_grad():
  for test_path in tqdm(test_images_filepaths): # 테스트 데이터셋 이용
    img = Image.open(test_path)
    _id = test_path.split('/')[-1].split('.')[1]
    transform = ImageTransform(size, mean, std)
    img = transform(img, phase='val') # 테스트 데이터셋 전처리 적용
    img = img.unsqueeze(0)
    img = img.to(device)

    model.eval()
    outputs = model(img)
    preds = F.softmax(outputs, dim=1)[:, 1].tolist()
    id_list.append(_id)
    pred_list.append(preds[0])

# 테스트 데이터셋의 예측 결과인 id와 레이블(label)을 데이터 프레임에 저장
res = pd.DataFrame({
    'id': id_list,
    'label': pred_list
})

res.sort_values(by='id', inplace=True)
res.reset_index(drop=True, inplace=True)

res.to_csv('/content/drive/MyDrive/Colab Notebooks/LeNet/LeNet', index=False)

/tmp/ipykernel_7466/532857518.py:7: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for test_path in tqdm(test_images_filepaths): # 테스트 데이터셋 이용


  0%|          | 0/10 [00:00<?, ?it/s]

In [36]:
res.head(10)

,id,label
0,109,0.581534
1,145,0.444593
2,15,0.657406
3,162,0.536508
4,167,0.531129
5,200,0.521810
6,210,0.648318
7,211,0.629954
8,213,0.478259
9,224,0.614203


In [37]:
class_ = classes = {0:'cat', 1:'dog'}

def display_image_grid(images_filepaths, predicted_labels=(), cols=5):
  rows = len(images_filepaths) // cols
  figure, ax = plt.subplots(nrows=rows, ncols=cols, figsize=(12, 6))

  for i, image_filepath in enumerate(images_filepaths):
    # OpenCV로 이미지 읽기(기본값은 BGR 순서)
    image = cv2.imread(image_filepath)
    # Matplotlib 출력을 위해 BGR -> RGB 순서로 색상 채널 변환
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # 결과 데이터프레임(res)의 id 값들 중 하나를 무작위로 선택
    a = random.choice(res['id'].values)

    # 선택된 id에 해당하는 예측 확률(label) 값을 데이터프레임에서 추출
    label = res.loc[res['id'] == a, 'label'].values[0]

    if label > 0.5:
      label = 1
    else:
      label = 0

    ax.ravel()[i].imshow(image)
    ax.ravel()[i].set_title(class_[label])
    ax.ravel()[i].set_axis_off()

  plt.tight_layout()
  plt.show()

In [ ]:
display_image_grid(test_images_filepaths)